In [ ]:
# Packages
import os
import re

# For downloading online NOAA data
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
import requests

# For data analysis
import pandas as pd
import numpy as np

In [ ]:
# Links of gzip storm data from NOAA website: https://www.ncei.noaa.gov/stormevents/ftp.jsp

def storm_data():
    # Web scrape NOAA weather data links
    storms_url = 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/'
    req = requests.get(storms_url)                  # access url webpage
    soup = BeautifulSoup(req.text, 'html.parser')   # parse thru HTML text of webpage

    # Find 2000s data.csv.gz filenames in <a href="link" > format 
    pattern = r'StormEvents_details-ftp_v1\.0_d20\d{2}_c\d{8}\.csv\.gz'   # 2000s filename pattern
    data_links = []
    for link in soup.find_all('a', attrs={'href': re.compile(pattern)}):
        year_data = link.get('href')
        full_link = str(storms_url) + str(year_data)
        data_links.append(full_link)
    return data_links


# Parallel download func for data
def download_files(data):
    # Create new directory for data
    data_dir = '../data'
    os.makedirs(data_dir, exist_ok=True)
    
    # Check url request for 'content-disposition' header to parse .gz filenames
    response = requests.get(data, stream=True)
    if 'content-disposition' in response.headers:
        content_disp = response.headers['content-disposition']
        file_name = content_disp.split('filename=')[1]
    else:
        file_name = data.split('/')[-1]
    
    # Write downloaded gzip data to data dir
    gz_name = os.path.join(data_dir, file_name)
    with open(gz_name, 'wb') as gz_file:
        gz_file.write(response.content)
    # print(f'Downloaded file to {gz_name}')


# Use ThreadPoolExecutor() to parallel download gzip files
with ThreadPoolExecutor() as executor:
    executor.map(download_files, storm_data())

In [ ]:
# Create pandas dataframes of data for each year (optional: state)
# Info about files: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/Storm-Data-Bulk-csv-Format.pdf 
def weather_df(year, *state):
    year: int   # set year as int
    
    # Look for selected year's data from data/ dir
    file_pattern = rf'StormEvents_details-ftp_v1\.0_d{year}_c\d{{8}}\.csv\.gz'
    gz_files = os.listdir('../data')
    try:
        match = [g for g in gz_files if re.search(file_pattern, g)][0]
    except IndexError:
        print('No matches found! Did you select a year between 2000 and 2026?')
    
    # Create curated df from match 
    df = pd.read_csv(f'../data/{match}', compression='gzip', header=0)
    df['BEGIN_DAY'] = df['BEGIN_DAY'].apply(lambda x: '0'+str(x) if len(str(x))<2 else str(x))
    df['BEGIN_DATE'] = df['BEGIN_YEARMONTH'].astype(str) + df['BEGIN_DAY']
    df['END_DAY'] = df['END_DAY'].apply(lambda x: '0'+str(x) if len(str(x))<2 else str(x))
    df['END_DATE'] = df['END_YEARMONTH'].astype(str) + df['END_DAY']
    
    # List of interested details
    details = ['BEGIN_DATE', 'END_DATE', 'STATE', 'CZ_NAME', 'EVENT_TYPE',
               'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT',
               'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'MAGNITUDE', 'TOR_F_SCALE',
               'TOR_LENGTH', 'TOR_WIDTH', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE']
    df = df[details]
    
    return df.head()

weather_df(2019)


,BEGIN_DATE,END_DATE,STATE,CZ_NAME,EVENT_TYPE,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,MAGNITUDE,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,EPISODE_NARRATIVE,EVENT_NARRATIVE
0,20190509,20190509,TEXAS,BEXAR,Flash Flood,0,0,0,0,0.00K,NaN,NaN,NaN,NaN,Thunderstorms developed along a cold front as ...,Thunderstorms produced heavy rain that led to ...
1,20190801,20190807,SOUTH DAKOTA,BROOKINGS,Flood,0,0,0,0,0.00K,NaN,NaN,NaN,NaN,Minor flooding slowly dwindled during early Au...,"A continuation of flooding from July, the Big ..."
2,20190925,20190925,ARIZONA,MARICOPA,Tornado,0,0,0,0,0.00K,NaN,EF0,0.67,50.0,Scattered thunderstorms developed over the cen...,Scattered thunderstorms developed across the c...
3,20190219,20190219,ARKANSAS,BOONE,Ice Storm,0,0,0,0,0.00K,NaN,NaN,NaN,NaN,"Rain was heavy at times on the 19th, and there...",One-quarter inch of freezing rain was measured...
4,20190219,20190219,ARKANSAS,BOONE,Ice Storm,0,0,0,0,0.00K,NaN,NaN,NaN,NaN,"Rain was heavy at times on the 19th, and there...",One-quarter inch of freezing rain was measured...
